# 📊 Mutual Fund Analysis Dashboard — Data Cleaning
**Step 2: Clean all raw data files with Pandas**

Files covered:
1. `all_schemes.csv`
2. `amfi_master.csv`
3. `aum_amfi_monthly.csv`
4. `aum_and_ter.csv`
5. `benchmark_data.csv`
6. `category_reference.csv`
7. `nav_history_raw.csv`
8. `nav_snapshots.csv`
9. `scheme_metadata.csv`
10. `scheme_profiles.csv`
11. `amapr2026repo.xls`

> ⚠️ **Safety rule throughout**: We NEVER drop rows unless we are 100% sure the row is truly unusable (e.g. full-row duplicates, or header rows mixed into data). For missing values we flag/fill, not drop.

In [241]:
import pandas as pd
import numpy as np

# ─── Set display options for comfortable inspection ───
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded ✅')

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG — edit these to control what gets collected
# ─────────────────────────────────────────────────────────────────────────────
INPUT_DIR   = "data/raw"

Libraries loaded ✅


---
## 1. `all_schemes.csv`
**Shape**: 37,601 rows × 4 cols  
**Issues found**:
- `isin_growth` → 29,214 nulls (78% missing) — many schemes don't have a growth ISIN, that's **valid data absence**, not an error
- `isin_div_reinvestment` → 33,490 nulls (89% missing) — same reason
- `scheme_name` → 274 rows with leading/trailing whitespace
- No duplicate rows or duplicate `scheme_code`s

In [242]:
df_schemes = pd.read_csv(f"{INPUT_DIR}/all_schemes.csv")
print('Shape before cleaning:', df_schemes.shape)
df_schemes.head(3)

Shape before cleaning: (37602, 4)


,scheme_code,scheme_name,isin_growth,isin_div_reinvestment
0,100027,Grindlays Super Saver Income Fund-GSSIF-Half Yearly Divi...,NaN,NaN
1,100028,Grindlays Super Saver Income Fund-GSSIF-Quaterly Dividend,NaN,NaN
2,100029,Grindlays Super Saver Income Fund-GSSIF-Growth,NaN,NaN


In [243]:
# ── 1a. Check null counts ──
print('Null counts:')
print(df_schemes.isnull().sum())

# NOTE: isin_growth and isin_div_reinvestment nulls are EXPECTED.
# Thousands of schemes are closed-ended, direct plans, or IDCW-only
# and simply don't have both ISINs. DO NOT drop these rows.

Null counts:
scheme_code                  0
scheme_name                  0
isin_growth              29214
isin_div_reinvestment    33491
dtype: int64


In [244]:
# ── 1b. Trim whitespace from scheme_name (274 rows affected) ──
df_schemes['scheme_name'] = df_schemes['scheme_name'].str.strip()

# Verify fix
print('Whitespace rows remaining:', (df_schemes['scheme_name'] != df_schemes['scheme_name'].str.strip()).sum())

Whitespace rows remaining: 0


In [245]:
# ── 1c. Confirm no duplicate scheme_codes ──
print('Duplicate scheme_code count:', df_schemes.duplicated(subset=['scheme_code']).sum())
# Expected: 0

Duplicate scheme_code count: 0


In [246]:
# ── 1d. Ensure scheme_code is int ──
df_schemes['scheme_code'] = df_schemes['scheme_code'].astype(int)

print('Shape after cleaning:', df_schemes.shape)
print(df_schemes.dtypes)
df_schemes.head(3)

Shape after cleaning: (37602, 4)
scheme_code              int64
scheme_name                str
isin_growth                str
isin_div_reinvestment      str
dtype: object


,scheme_code,scheme_name,isin_growth,isin_div_reinvestment
0,100027,Grindlays Super Saver Income Fund-GSSIF-Half Yearly Divi...,NaN,NaN
1,100028,Grindlays Super Saver Income Fund-GSSIF-Quaterly Dividend,NaN,NaN
2,100029,Grindlays Super Saver Income Fund-GSSIF-Growth,NaN,NaN


---
## 2. `amfi_master.csv`
**Shape**: 14,366 rows × 8 cols  
**Issues found**:
- `isin_idcw` → 9,876 rows contain `'-'` (a dash string, not a true null). These need to be replaced with `NaN`
- `nav_date` → stored as string, needs conversion to `datetime`
- `broad_category` → contains the full AMFI verbose string like `'Open Ended Schemes(Debt Scheme - Banking and...'`, should be kept as-is but can be trimmed
- No missing values, no duplicate `scheme_code`s

In [247]:
df_amfi = pd.read_csv(f"{INPUT_DIR}/amfi_master.csv")
print('Shape before cleaning:', df_amfi.shape)
df_amfi.head(3)

Shape before cleaning: (14367, 8)


,scheme_code,isin_growth,isin_idcw,scheme_name,nav,nav_date,amc_name,broad_category
0,119551,INF209KA12Z1,INF209KA13Z9,Aditya Birla Sun Life Banking & PSU Debt Fund - DIRECT ...,104.4216,2026-05-19,Aditya Birla Sun Life Mutual Fund,Open Ended Schemes(Debt Scheme - Banking and PSU Fund)
1,119552,INF209K01YM2,-,Aditya Birla Sun Life Banking & PSU Debt Fund - DIRECT ...,115.9453,2026-05-19,Aditya Birla Sun Life Mutual Fund,Open Ended Schemes(Debt Scheme - Banking and PSU Fund)
2,119553,INF209K01YO8,-,Aditya Birla Sun Life Banking & PSU Debt Fund - Direct ...,103.7089,2026-05-19,Aditya Birla Sun Life Mutual Fund,Open Ended Schemes(Debt Scheme - Banking and PSU Fund)


In [248]:
# ── 2a. Replace '-' dash strings in ISIN columns with NaN ──
# '-' is used as a placeholder when no ISIN exists; it must become NaN, not stay as a string
df_amfi['isin_growth'] = df_amfi['isin_growth'].replace('-', np.nan)
df_amfi['isin_idcw']   = df_amfi['isin_idcw'].replace('-', np.nan)

print('isin_growth nulls after fix:', df_amfi['isin_growth'].isnull().sum())
print('isin_idcw nulls after fix:  ', df_amfi['isin_idcw'].isnull().sum())

isin_growth nulls after fix: 792
isin_idcw nulls after fix:   9877


In [249]:
# ── 2b. Convert nav_date from string to datetime ──
df_amfi['nav_date'] = pd.to_datetime(df_amfi['nav_date'], dayfirst=True, errors='coerce')

# Check for any failed conversions
failed_dates = df_amfi['nav_date'].isnull().sum()
print(f'nav_date parse failures: {failed_dates}')

nav_date parse failures: 0


C:\Users\krish\AppData\Local\Temp\ipykernel_1688\24344438.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_amfi['nav_date'] = pd.to_datetime(df_amfi['nav_date'], dayfirst=True, errors='coerce')


In [250]:
# ── 2c. Trim whitespace from all string columns ──
for col in ['isin_growth', 'isin_idcw', 'scheme_name', 'amc_name', 'broad_category']:
    df_amfi[col] = df_amfi[col].str.strip()

# ── 2d. Confirm no duplicate scheme_codes ──
print('Duplicate scheme_code:', df_amfi.duplicated(subset=['scheme_code']).sum())

print('Shape after cleaning:', df_amfi.shape)
print(df_amfi.dtypes)
df_amfi.head(3)

Duplicate scheme_code: 0
Shape after cleaning: (14367, 8)
scheme_code                int64
isin_growth                  str
isin_idcw                    str
scheme_name                  str
nav                      float64
nav_date          datetime64[us]
amc_name                     str
broad_category               str
dtype: object


,scheme_code,isin_growth,isin_idcw,scheme_name,nav,nav_date,amc_name,broad_category
0,119551,INF209KA12Z1,INF209KA13Z9,Aditya Birla Sun Life Banking & PSU Debt Fund - DIRECT ...,104.4216,2026-05-19,Aditya Birla Sun Life Mutual Fund,Open Ended Schemes(Debt Scheme - Banking and PSU Fund)
1,119552,INF209K01YM2,NaN,Aditya Birla Sun Life Banking & PSU Debt Fund - DIRECT ...,115.9453,2026-05-19,Aditya Birla Sun Life Mutual Fund,Open Ended Schemes(Debt Scheme - Banking and PSU Fund)
2,119553,INF209K01YO8,NaN,Aditya Birla Sun Life Banking & PSU Debt Fund - Direct ...,103.7089,2026-05-19,Aditya Birla Sun Life Mutual Fund,Open Ended Schemes(Debt Scheme - Banking and PSU Fund)


---
## 3. `aum_amfi_monthly.csv`
**Shape**: 91 rows × 11 cols  
**Issues found** — ⚠️ This file is a RAW AMFI PDF-scraped table, NOT a clean tabular CSV:
- Row 0 = original CSV headers row (should be skipped)
- Row 1 = report title `'Monthly Report for April 2026'` — a meta-header row
- Row 2 = actual real column headers
- Rows 3, 4 = category group label rows (`'A - Open ended Schemes'`, `'I - Income/...'`) — no data
- All column names are `unnamed:_0` through `unnamed:_10` because the file was read without skipping header rows
- 18 full-row duplicates (blank spacer rows)
- Last 3 rows = footnote/disclaimer text, not data
- All values are currently `str` dtype; numeric columns need conversion

> **Strategy**: Re-read with correct header row, strip group/blank/footnote rows, convert numerics.

In [251]:
# First look at the raw structure
df_aum_raw = pd.read_csv(f"{INPUT_DIR}/aum_amfi_monthly.csv", header=None)
print('Raw shape:', df_aum_raw.shape)
df_aum_raw.head(6)

Raw shape: (92, 11)


,0,1,2,3,4,5,6,7,8,9,10
0,unnamed:_0,unnamed:_1,unnamed:_2,unnamed:_3,unnamed:_4,unnamed:_5,unnamed:_6,unnamed:_7,unnamed:_8,unnamed:_9,unnamed:_10
1,Monthly Report for April 2026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sr,Scheme Name,"No. of Schemes as on April 30, 2026","No. of Folios as on April 30, 2026",Funds Mobilized for the month of April 2026 (INR in crore),Repurchase/Redemption for the month of April 2026 (INR i...,Net Inflow (+ve)/Outflow (-ve) for the month of April 20...,"Net Assets Under Management as on April 30, 2026 (INR in...",Average Net Assets Under Management for the month April ...,"No. of segregated portfolios created as on April 30, 2026",Net Assets Under Management in segregated portfolio as o...
3,A,Open ended Schemes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,I,Income/Debt Oriented Schemes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,i,Overnight Fund,37,764557,564711.91656964,533291.46887816,31420.44769148,104919.8565146,118330.62323408,0,0


In [252]:
# ── 3a. Re-read correctly: row index 2 (3rd row) is the actual header ──
# skiprows=[0,1] skips: the pandas-generated header row & the title row
df_aum = pd.read_csv(f"{INPUT_DIR}/aum_amfi_monthly.csv", skiprows=[0, 1], header=0)
print('Shape after correct header:', df_aum.shape)
df_aum.head(5)

Shape after correct header: (89, 11)


,Sr,Scheme Name,"No. of Schemes as on April 30, 2026","No. of Folios as on April 30, 2026",Funds Mobilized for the month of April 2026 (INR in crore),Repurchase/Redemption for the month of April 2026 (INR in crore),Net Inflow (+ve)/Outflow (-ve) for the month of April 2026 (INR in crore),"Net Assets Under Management as on April 30, 2026 (INR in crore)",Average Net Assets Under Management for the month April 2026 (INR in crore),"No. of segregated portfolios created as on April 30, 2026","Net Assets Under Management in segregated portfolio as on April 30, 2026 (INR in crore)"
0,A,Open ended Schemes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,I,Income/Debt Oriented Schemes,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,i,Overnight Fund,37,764557.0000,564711.9166,533291.4689,31420.4477,104919.8565,118330.6232,0.0000,0.0000
3,ii,Liquid Fund,42,2907839.0000,583842.2233,418737.5501,165104.6732,635970.6884,668559.9171,0.0000,0.0000
4,iii,Ultra Short Duration Fund,25,787200.0000,37998.0869,22346.2209,15651.8660,129712.0615,126217.9457,0.0000,0.0000


In [253]:
# ── 3b. Rename columns to clean, short names ──
df_aum.columns = [
    'sr_no', 'scheme_name', 'num_schemes', 'num_folios',
    'funds_mobilized_cr', 'repurchase_redemption_cr', 'net_inflow_outflow_cr',
    'aum_cr', 'avg_aum_cr', 'segregated_portfolios', 'segregated_aum_cr'
]
print('Columns renamed:')
print(df_aum.columns.tolist())

Columns renamed:
['sr_no', 'scheme_name', 'num_schemes', 'num_folios', 'funds_mobilized_cr', 'repurchase_redemption_cr', 'net_inflow_outflow_cr', 'aum_cr', 'avg_aum_cr', 'segregated_portfolios', 'segregated_aum_cr']


In [254]:
# ── 3c. Drop rows that are category group labels (no numeric data) ──
# These are rows like 'A - Open ended Schemes', 'I - Income/Debt Oriented Schemes' etc.
# They have NaN in numeric columns. But we must be careful — we only drop
# rows where BOTH scheme_name is a category label AND all numeric cols are NaN.

numeric_cols = ['num_schemes', 'funds_mobilized_cr', 'aum_cr']

# Rows where all three key numeric columns are NaN = group header / blank / footnote rows
mask_empty = df_aum[numeric_cols].isnull().all(axis=1)
print(f'Group/blank/footnote rows to remove: {mask_empty.sum()}')
print(df_aum[mask_empty][['sr_no','scheme_name']].to_string())

Group/blank/footnote rows to remove: 28
   sr_no                                                                                                                          scheme_name
0      A                                                                                                                   Open ended Schemes
1      I                                                                                                         Income/Debt Oriented Schemes
19   NaN                                                                                                                                  NaN
20    II                                                                                                       Growth/Equity Oriented Schemes
33   NaN                                                                                                                                  NaN
34   III                                                                                                    

In [255]:
# ── 3d. Keep only rows that have actual data ──
df_aum = df_aum[~mask_empty].reset_index(drop=True)
print('Shape after removing group/blank rows:', df_aum.shape)

Shape after removing group/blank rows: (61, 11)


In [256]:
# ── 3e. Convert numeric columns (they came in as strings) ──
cols_to_numeric = [
    'num_schemes', 'num_folios', 'funds_mobilized_cr',
    'repurchase_redemption_cr', 'net_inflow_outflow_cr',
    'aum_cr', 'avg_aum_cr', 'segregated_portfolios', 'segregated_aum_cr'
]
for col in cols_to_numeric:
    df_aum[col] = pd.to_numeric(df_aum[col], errors='coerce')

print(df_aum.dtypes)

sr_no                           str
scheme_name                     str
num_schemes                 float64
num_folios                  float64
funds_mobilized_cr          float64
repurchase_redemption_cr    float64
net_inflow_outflow_cr       float64
aum_cr                      float64
avg_aum_cr                  float64
segregated_portfolios       float64
segregated_aum_cr           float64
dtype: object


In [257]:
# ── 3f. Trim whitespace in string columns ──
df_aum['scheme_name'] = df_aum['scheme_name'].str.strip()
df_aum['sr_no']       = df_aum['sr_no'].astype(str).str.strip()

print('Shape after full cleaning:', df_aum.shape)
df_aum.head(5)

Shape after full cleaning: (61, 11)


,sr_no,scheme_name,num_schemes,num_folios,funds_mobilized_cr,repurchase_redemption_cr,net_inflow_outflow_cr,aum_cr,avg_aum_cr,segregated_portfolios,segregated_aum_cr
0,i,Overnight Fund,37.0000,764557.0000,564711.9166,533291.4689,31420.4477,104919.8565,118330.6232,0.0000,0.0000
1,ii,Liquid Fund,42.0000,2907839.0000,583842.2233,418737.5501,165104.6732,635970.6884,668559.9171,0.0000,0.0000
2,iii,Ultra Short Duration Fund,25.0000,787200.0000,37998.0869,22346.2209,15651.8660,129712.0615,126217.9457,0.0000,0.0000
3,iv,Low Duration Fund,25.0000,852755.0000,21870.9658,14777.7042,7093.2616,138398.0408,137868.6230,0.0000,0.0000
4,v,Money Market Fund,27.0000,534751.0000,112839.0652,92196.4792,20642.5860,334924.3494,329745.5267,1.0000,0.0000


---
## 5. `benchmark_data.csv`
**Shape**: 1,812 rows × 9 cols  
**Issues found**:
- `date` → string, needs `datetime` conversion
- Nulls in index columns: `Nifty 50` (86), `Nifty 500` (93), `Nifty Next 50` (95), `BSE Sensex` (90), `Nifty Bank` (87), `India 10Y Bond` (54)
- These nulls = **market holidays or data gaps**, NOT errors — DO NOT drop rows
- `Nifty Midcap 150` and `Nifty Smallcap 250` have 0 nulls
- No duplicate dates
- Should be sorted by date

In [258]:
df_bench = pd.read_csv(f"{INPUT_DIR}/benchmark_data.csv")
print('Shape before cleaning:', df_bench.shape)
df_bench.head(3)

Shape before cleaning: (1812, 9)


,date,Nifty 50,Nifty 500,Nifty Midcap 150,Nifty Smallcap 250,Nifty Next 50,BSE Sensex,Nifty Bank,India 10Y Bond
0,2019-05-22,11737.9004,9598.1875,14298.8496,17863.5508,26847.5410,39110.2109,30526.4453,2.3200
1,2019-05-23,11657.0498,9552.7373,14298.8496,17863.5508,26946.5879,38811.3906,30408.7461,2.3030
2,2019-05-24,11844.0996,9722.0371,14298.8496,17863.5508,27572.1270,39434.7188,31212.1875,2.2880


In [259]:
# ── 5a. Convert date column to datetime ──
df_bench['date'] = pd.to_datetime(df_bench['date'], errors='coerce')

failed = df_bench['date'].isnull().sum()
print(f'Date parse failures: {failed}')
print('Date range:', df_bench['date'].min(), 'to', df_bench['date'].max())

Date parse failures: 0
Date range: 2019-05-22 00:00:00 to 2026-05-19 00:00:00


In [260]:
# ── 5b. Sort by date ──
df_bench = df_bench.sort_values('date').reset_index(drop=True)
print('Sorted by date ✅')

Sorted by date ✅


In [261]:
# ── 5c. Check for duplicate dates ──
print('Duplicate dates:', df_bench.duplicated(subset=['date']).sum())
# Expected: 0

Duplicate dates: 0


In [262]:
# ── 5d. Document null counts per index (these are holiday gaps — keep as NaN) ──
print('Null counts per index column:')
print(df_bench.isnull().sum())
# NaN here means trading was closed that day for that specific index — valid absence of data
# Do NOT forward-fill here — price imputation should be a deliberate analysis decision later

Null counts per index column:
date                   0
Nifty 50              87
Nifty 500             94
Nifty Midcap 150       0
Nifty Smallcap 250     0
Nifty Next 50         96
BSE Sensex            91
Nifty Bank            88
India 10Y Bond        54
dtype: int64


In [263]:
# ── 5e. Ensure all index columns are float ──
index_cols = ['Nifty 50', 'Nifty 500', 'Nifty Midcap 150', 'Nifty Smallcap 250',
              'Nifty Next 50', 'BSE Sensex', 'Nifty Bank', 'India 10Y Bond']
for col in index_cols:
    df_bench[col] = pd.to_numeric(df_bench[col], errors='coerce')

print('Shape after cleaning:', df_bench.shape)
print(df_bench.dtypes)

Shape after cleaning: (1812, 9)
date                  datetime64[us]
Nifty 50                     float64
Nifty 500                    float64
Nifty Midcap 150             float64
Nifty Smallcap 250           float64
Nifty Next 50                float64
BSE Sensex                   float64
Nifty Bank                   float64
India 10Y Bond               float64
dtype: object


---
## 6. `category_reference.csv`
**Shape**: 41 rows × 4 cols  
**Issues found**:
- Zero nulls, zero duplicate rows ✅
- `broad_category` has 36 duplicates — **this is correct behaviour**: one broad category (e.g. 'Equity Scheme') has many sub-categories
- `sub_category` is the unique identifier here — 0 duplicates ✅
- Minor: string columns may have inconsistent spacing — trim to be safe

In [264]:
df_cat = pd.read_csv(f"{INPUT_DIR}/category_reference.csv")
print('Shape:', df_cat.shape)
df_cat

Shape: (41, 4)


,broad_category,sub_category,benchmark_index,risk_level
0,Equity Scheme,Large Cap Fund,Nifty 100,Moderately High
1,Equity Scheme,Large & Mid Cap Fund,Nifty LargeMidcap 250,Moderately High
2,Equity Scheme,Mid Cap Fund,Nifty Midcap 150,High
3,Equity Scheme,Small Cap Fund,Nifty Smallcap 250,Very High
4,Equity Scheme,Multi Cap Fund,Nifty 500,High
5,Equity Scheme,Flexi Cap Fund,Nifty 500,Moderately High
6,Equity Scheme,ELSS,Nifty 500,High
7,Equity Scheme,Sectoral / Thematic Fund,Nifty 500,Very High
8,Equity Scheme,Focused Fund,Nifty 500,High
9,Equity Scheme,Dividend Yield Fund,Nifty Dividend Opp 50,Moderately High


In [265]:
# ── 6a. Trim whitespace on all string columns ──
for col in df_cat.select_dtypes(include='object').columns:
    df_cat[col] = df_cat[col].str.strip()

# ── 6b. Verify sub_category is unique (the true PK for this lookup table) ──
print('Duplicate sub_category:', df_cat.duplicated(subset=['sub_category']).sum())

# ── 6c. Check null counts ──
print('Null counts:')
print(df_cat.isnull().sum())

print('\nClean ✅')
df_cat.head(5)

Duplicate sub_category: 0
Null counts:
broad_category     0
sub_category       0
benchmark_index    0
risk_level         0
dtype: int64

Clean ✅


C:\Users\krish\AppData\Local\Temp\ipykernel_1688\1313942647.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df_cat.select_dtypes(include='object').columns:


,broad_category,sub_category,benchmark_index,risk_level
0,Equity Scheme,Large Cap Fund,Nifty 100,Moderately High
1,Equity Scheme,Large & Mid Cap Fund,Nifty LargeMidcap 250,Moderately High
2,Equity Scheme,Mid Cap Fund,Nifty Midcap 150,High
3,Equity Scheme,Small Cap Fund,Nifty Smallcap 250,Very High
4,Equity Scheme,Multi Cap Fund,Nifty 500,High


---
## 7. `nav_history_raw.csv`
**Shape**: 22,692 rows × 4 cols  
**Issues found**:
- Zero nulls ✅
- `scheme_code` has 22,679 "duplicates" → **correct**, one scheme has many date rows
- `date` has 20,553 "duplicates" → **correct**, same date has rows for different schemes
- Composite key `(scheme_code, date)` = 0 duplicates ✅ — this is the real uniqueness check
- `date` stored as string → needs `datetime` conversion
- Should be sorted by `(scheme_code, date)` for time-series operations

In [266]:
df_nav = pd.read_csv(f"{INPUT_DIR}/nav_history_raw.csv")
print('Shape before cleaning:', df_nav.shape)
df_nav.head(3)

Shape before cleaning: (24052, 4)


,date,nav,scheme_code,fund_name
0,2019-05-23,31.3400,120465,Axis Large Cap Fund - Direct Plan - Growth
1,2019-05-24,31.7500,120465,Axis Large Cap Fund - Direct Plan - Growth
2,2019-05-27,31.8500,120465,Axis Large Cap Fund - Direct Plan - Growth


In [267]:
# ── 7a. Verify composite key uniqueness (scheme_code + date) ──
dups = df_nav.duplicated(subset=['scheme_code', 'date']).sum()
print(f'Duplicate (scheme_code, date) pairs: {dups}')
# If > 0, investigate before deciding to drop

Duplicate (scheme_code, date) pairs: 0


In [268]:
# ── 7b. Convert date to datetime ──
df_nav['date'] = pd.to_datetime(df_nav['date'], errors='coerce')

failed = df_nav['date'].isnull().sum()
print(f'Date parse failures: {failed}')
print('Date range:', df_nav['date'].min(), 'to', df_nav['date'].max())

Date parse failures: 0
Date range: 2019-05-23 00:00:00 to 2026-05-19 00:00:00


In [269]:
# ── 7c. Sort by scheme_code then date (essential for returns calculation) ──
df_nav = df_nav.sort_values(['scheme_code', 'date']).reset_index(drop=True)
print('Sorted by scheme_code + date ✅')

Sorted by scheme_code + date ✅


In [270]:
# ── 7d. Validate NAV values — no NAV should be zero or negative ──
invalid_nav = df_nav[df_nav['nav'] <= 0]
print(f'Rows with NAV ≤ 0: {len(invalid_nav)}')
if len(invalid_nav) > 0:
    print(invalid_nav)
    # These should be investigated — could be data entry errors

Rows with NAV ≤ 0: 0


In [271]:
# ── 7e. Trim whitespace in fund_name ──
df_nav['fund_name'] = df_nav['fund_name'].str.strip()

print('Shape after cleaning:', df_nav.shape)
print(df_nav.dtypes)
df_nav.head(3)

Shape after cleaning: (24052, 4)
date           datetime64[us]
nav                   float64
scheme_code             int64
fund_name                 str
dtype: object


,date,nav,scheme_code,fund_name
0,2019-05-23,139.2700,118275,CANARA ROBECO FLEXICAP FUND - DIRECT PLAN - GROWTH OPTION
1,2019-05-24,141.9700,118275,CANARA ROBECO FLEXICAP FUND - DIRECT PLAN - GROWTH OPTION
2,2019-05-27,143.9500,118275,CANARA ROBECO FLEXICAP FUND - DIRECT PLAN - GROWTH OPTION


---
## 8. `nav_snapshots.csv`
**Shape**: 13 rows × 9 cols  
**Issues found**:
- Zero nulls ✅
- Zero duplicates ✅
- All NAV columns are `float64` — correct ✅
- Minor: trim `fund_name` whitespace
- **Logical check**: `nav_52w_high ≥ nav_today ≥ nav_52w_low` should hold; worth verifying

In [272]:
df_snap = pd.read_csv(f"{INPUT_DIR}/nav_snapshots.csv")
print('Shape:', df_snap.shape)
df_snap

Shape: (14, 9)


,scheme_code,fund_name,nav_today,nav_1m_ago,nav_3m_ago,nav_6m_ago,nav_1y_ago,nav_3y_ago,nav_5y_ago
0,120465,Axis Large Cap Fund - Direct Plan - Growth,65.8200,67.7800,71.6700,71.8700,68.1300,49.0600,44.2600
1,118825,Mirae Asset Large Cap Fund - Direct Plan - Growth,122.0200,125.2210,131.5510,132.9900,123.6650,88.7050,73.0860
2,122639,Parag Parikh Flexi Cap Fund - Direct Plan - Growth,90.6579,91.6716,93.4081,94.3086,89.6835,56.7787,43.1322
3,118275,CANARA ROBECO FLEXICAP FUND - DIRECT PLAN - GROWTH OPTION,371.9000,378.3100,391.9000,396.1100,366.9200,251.2800,204.2300
4,120505,Axis Midcap Fund - Direct Plan - Growth,135.4500,134.1600,133.5100,134.7700,124.7800,78.7000,63.2700
5,119775,Kotak Midcap Fund - Direct Plan - Growth,161.5860,160.9790,160.8320,160.7240,146.0440,87.9510,66.5450
6,125497,SBI Small Cap Fund - Direct Plan - Growth,189.5553,191.0583,188.2128,196.5280,190.1620,130.2383,95.3914
7,118777,Nippon India Small Cap Fund - Direct Plan Growth,193.4247,189.3466,185.1051,191.1666,182.2795,109.4846,71.4140
8,120716,UTI Nifty 50 Index Fund - Growth Option - Direct,165.2113,170.3089,180.5280,182.0266,171.0593,123.9877,101.2594
9,119063,HDFC Nifty 50 Index Fund - Direct Plan,229.7357,236.8352,251.0643,253.2021,237.9790,172.6182,141.0807


In [273]:
# ── 8a. Trim whitespace in fund_name ──
df_snap['fund_name'] = df_snap['fund_name'].str.strip()

# ── 8b. Validate historical NAVs are in ascending order (older = smaller for equity) ──
# This is a soft check — not all funds grow, so just look for obvious issues
print('Null check:')
print(df_snap.isnull().sum())

# ── 8c. Validate: nav_today should be positive ──
print('\nNegative/zero nav_today:', (df_snap['nav_today'] <= 0).sum())

Null check:
scheme_code    0
fund_name      0
nav_today      0
nav_1m_ago     0
nav_3m_ago     0
nav_6m_ago     0
nav_1y_ago     0
nav_3y_ago     0
nav_5y_ago     0
dtype: int64

Negative/zero nav_today: 0


In [274]:
# ── 8d. Check for duplicate scheme_code ──
print('Duplicate scheme_code:', df_snap.duplicated(subset=['scheme_code']).sum())

print('\nClean ✅')
df_snap.head(3)

Duplicate scheme_code: 0

Clean ✅


,scheme_code,fund_name,nav_today,nav_1m_ago,nav_3m_ago,nav_6m_ago,nav_1y_ago,nav_3y_ago,nav_5y_ago
0,120465,Axis Large Cap Fund - Direct Plan - Growth,65.8200,67.7800,71.6700,71.8700,68.1300,49.0600,44.2600
1,118825,Mirae Asset Large Cap Fund - Direct Plan - Growth,122.0200,125.2210,131.5510,132.9900,123.6650,88.7050,73.0860
2,122639,Parag Parikh Flexi Cap Fund - Direct Plan - Growth,90.6579,91.6716,93.4081,94.3086,89.6835,56.7787,43.1322


---
## 9. `scheme_metadata.csv`
**Shape**: 14 rows × 7 cols  
**Issues found**:
- Row for `scheme_code = 119260` has nulls in ALL non-key columns (`scheme_name`, `fund_house`, `scheme_type`, `broad_category`, `sub_category`, `full_category`) — this scheme could not be matched from AMFI data
- Do NOT drop this row — keep it flagged as incomplete. The scheme exists in other files (nav_history, aum_and_ter) and dropping it would break joins
- No duplicate `scheme_code`s

In [275]:
df_meta = pd.read_csv(f"{INPUT_DIR}/scheme_metadata.csv")
print('Shape before cleaning:', df_meta.shape)
df_meta

Shape before cleaning: (14, 7)


,scheme_code,scheme_name,fund_house,scheme_type,broad_category,sub_category,full_category
0,120465,Axis Large Cap Fund - Direct Plan - Growth,Axis Mutual Fund,Open Ended Schemes,Equity Scheme,Large Cap Fund,Equity Scheme - Large Cap Fund
1,118825,Mirae Asset Large Cap Fund - Direct Plan - Growth,Mirae Asset Mutual Fund,Open Ended Schemes,Equity Scheme,Large Cap Fund,Equity Scheme - Large Cap Fund
2,122639,Parag Parikh Flexi Cap Fund - Direct Plan - Growth,PPFAS Mutual Fund,Open Ended Schemes,Equity Scheme,Flexi Cap Fund,Equity Scheme - Flexi Cap Fund
3,118275,CANARA ROBECO FLEXICAP FUND - DIRECT PLAN - GROWTH OPTION,Canara Robeco Mutual Fund,Open Ended Schemes,Equity Scheme,Flexi Cap Fund,Equity Scheme - Flexi Cap Fund
4,120505,Axis Midcap Fund - Direct Plan - Growth,Axis Mutual Fund,Open Ended Schemes,Equity Scheme,Mid Cap Fund,Equity Scheme - Mid Cap Fund
5,119775,Kotak Midcap Fund - Direct Plan - Growth,Kotak Mahindra Mutual Fund,Open Ended Schemes,Equity Scheme,Mid Cap Fund,Equity Scheme - Mid Cap Fund
6,125497,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Open Ended Schemes,Equity Scheme,Small Cap Fund,Equity Scheme - Small Cap Fund
7,118777,Nippon India Small Cap Fund - Direct Plan Growth Plan - ...,Nippon India Mutual Fund,Open Ended Schemes,Equity Scheme,Small Cap Fund,Equity Scheme - Small Cap Fund
8,120716,UTI Nifty 50 Index Fund - Growth Option- Direct,UTI Mutual Fund,Open Ended Schemes,Other Scheme,Index Funds,Other Scheme - Index Funds
9,119063,HDFC Nifty 50 Index Fund - Direct Plan,HDFC Mutual Fund,Open Ended Schemes,Other Scheme,Index Funds,Other Scheme - Index Funds


In [276]:
# ── 9a. Identify the incomplete row ──
incomplete = df_meta[df_meta.isnull().any(axis=1)]
print('Rows with any null values:')
print(incomplete)

Rows with any null values:
Empty DataFrame
Columns: [scheme_code, scheme_name, fund_house, scheme_type, broad_category, sub_category, full_category]
Index: []


In [277]:
# ── 9b. Flag incomplete rows with a boolean column ──
df_meta['metadata_complete'] = ~df_meta.isnull().any(axis=1)

print('Complete rows:', df_meta['metadata_complete'].sum())
print('Incomplete rows:', (~df_meta['metadata_complete']).sum())

Complete rows: 14
Incomplete rows: 0


In [278]:
# ── 9c. Trim whitespace on all string columns ──
for col in ['scheme_name', 'fund_house', 'scheme_type', 'broad_category', 'sub_category', 'full_category']:
    df_meta[col] = df_meta[col].str.strip()

# ── 9d. Confirm no duplicate scheme_code ──
print('Duplicate scheme_code:', df_meta.duplicated(subset=['scheme_code']).sum())

print('Shape after cleaning:', df_meta.shape)
print(df_meta.dtypes)
df_meta

Duplicate scheme_code: 0
Shape after cleaning: (14, 8)
scheme_code          int64
scheme_name            str
fund_house             str
scheme_type            str
broad_category         str
sub_category           str
full_category          str
metadata_complete     bool
dtype: object


,scheme_code,scheme_name,fund_house,scheme_type,broad_category,sub_category,full_category,metadata_complete
0,120465,Axis Large Cap Fund - Direct Plan - Growth,Axis Mutual Fund,Open Ended Schemes,Equity Scheme,Large Cap Fund,Equity Scheme - Large Cap Fund,True
1,118825,Mirae Asset Large Cap Fund - Direct Plan - Growth,Mirae Asset Mutual Fund,Open Ended Schemes,Equity Scheme,Large Cap Fund,Equity Scheme - Large Cap Fund,True
2,122639,Parag Parikh Flexi Cap Fund - Direct Plan - Growth,PPFAS Mutual Fund,Open Ended Schemes,Equity Scheme,Flexi Cap Fund,Equity Scheme - Flexi Cap Fund,True
3,118275,CANARA ROBECO FLEXICAP FUND - DIRECT PLAN - GROWTH OPTION,Canara Robeco Mutual Fund,Open Ended Schemes,Equity Scheme,Flexi Cap Fund,Equity Scheme - Flexi Cap Fund,True
4,120505,Axis Midcap Fund - Direct Plan - Growth,Axis Mutual Fund,Open Ended Schemes,Equity Scheme,Mid Cap Fund,Equity Scheme - Mid Cap Fund,True
5,119775,Kotak Midcap Fund - Direct Plan - Growth,Kotak Mahindra Mutual Fund,Open Ended Schemes,Equity Scheme,Mid Cap Fund,Equity Scheme - Mid Cap Fund,True
6,125497,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Open Ended Schemes,Equity Scheme,Small Cap Fund,Equity Scheme - Small Cap Fund,True
7,118777,Nippon India Small Cap Fund - Direct Plan Growth Plan - ...,Nippon India Mutual Fund,Open Ended Schemes,Equity Scheme,Small Cap Fund,Equity Scheme - Small Cap Fund,True
8,120716,UTI Nifty 50 Index Fund - Growth Option- Direct,UTI Mutual Fund,Open Ended Schemes,Other Scheme,Index Funds,Other Scheme - Index Funds,True
9,119063,HDFC Nifty 50 Index Fund - Direct Plan,HDFC Mutual Fund,Open Ended Schemes,Other Scheme,Index Funds,Other Scheme - Index Funds,True


---
## 10. `scheme_profiles.csv`
**Shape**: 13 rows × 10 cols  
**Issues found**:
- Zero nulls ✅
- Zero duplicates ✅
- `inception_date` and `latest_nav_date` stored as strings → need `datetime` conversion
- **Logical check**: `nav_52w_high` should be ≥ `nav_52w_low`; worth verifying
- `fund_age_years` should be positive; verify
- `total_nav_days` should be > 0; verify

In [279]:
df_prof = pd.read_csv(f"{INPUT_DIR}/scheme_profiles.csv")
print('Shape before cleaning:', df_prof.shape)
df_prof.head(3)

Shape before cleaning: (14, 10)


,scheme_code,fund_name,inception_date,fund_age_years,latest_nav,latest_nav_date,nav_52w_high,nav_52w_low,nav_52w_change_pct,total_nav_days
0,120465,Axis Large Cap Fund - Direct Plan - Growth,2019-05-23,7.0000,65.8200,2026-05-19,72.3800,62.2400,5.7500,1728
1,118825,Mirae Asset Large Cap Fund - Direct Plan - Growth,2019-05-23,7.0000,122.0200,2026-05-19,134.2340,113.1210,7.8700,1721
2,122639,Parag Parikh Flexi Cap Fund - Direct Plan - Growth,2019-05-23,7.0000,90.6579,2026-05-18,95.7732,85.2599,6.3300,1719


In [280]:
# ── 10a. Convert date columns from string to datetime ──
df_prof['inception_date']    = pd.to_datetime(df_prof['inception_date'], dayfirst=True, errors='coerce')
df_prof['latest_nav_date']   = pd.to_datetime(df_prof['latest_nav_date'], dayfirst=True, errors='coerce')

print('Date parse failures:')
print('  inception_date:', df_prof['inception_date'].isnull().sum())
print('  latest_nav_date:', df_prof['latest_nav_date'].isnull().sum())

Date parse failures:
  inception_date: 0
  latest_nav_date: 0


C:\Users\krish\AppData\Local\Temp\ipykernel_1688\576377804.py:2: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_prof['inception_date']    = pd.to_datetime(df_prof['inception_date'], dayfirst=True, errors='coerce')
C:\Users\krish\AppData\Local\Temp\ipykernel_1688\576377804.py:3: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_prof['latest_nav_date']   = pd.to_datetime(df_prof['latest_nav_date'], dayfirst=True, errors='coerce')


In [281]:
# ── 10b. Logical validation: 52-week high should be ≥ 52-week low ──
invalid_range = df_prof[df_prof['nav_52w_high'] < df_prof['nav_52w_low']]
print(f'Rows where 52w_high < 52w_low: {len(invalid_range)}')
if len(invalid_range) > 0:
    print(invalid_range[['scheme_code', 'fund_name', 'nav_52w_high', 'nav_52w_low']])

Rows where 52w_high < 52w_low: 0


In [282]:
# ── 10c. Validate fund_age_years and total_nav_days are positive ──
print('fund_age_years ≤ 0:', (df_prof['fund_age_years'] <= 0).sum())
print('total_nav_days ≤ 0:', (df_prof['total_nav_days'] <= 0).sum())

fund_age_years ≤ 0: 0
total_nav_days ≤ 0: 0


In [283]:
# ── 10d. Trim fund_name whitespace ──
df_prof['fund_name'] = df_prof['fund_name'].str.strip()

print('Shape after cleaning:', df_prof.shape)
print(df_prof.dtypes)
df_prof.head(3)

Shape after cleaning: (14, 10)
scheme_code                    int64
fund_name                        str
inception_date        datetime64[us]
fund_age_years               float64
latest_nav                   float64
latest_nav_date       datetime64[us]
nav_52w_high                 float64
nav_52w_low                  float64
nav_52w_change_pct           float64
total_nav_days                 int64
dtype: object


,scheme_code,fund_name,inception_date,fund_age_years,latest_nav,latest_nav_date,nav_52w_high,nav_52w_low,nav_52w_change_pct,total_nav_days
0,120465,Axis Large Cap Fund - Direct Plan - Growth,2019-05-23,7.0000,65.8200,2026-05-19,72.3800,62.2400,5.7500,1728
1,118825,Mirae Asset Large Cap Fund - Direct Plan - Growth,2019-05-23,7.0000,122.0200,2026-05-19,134.2340,113.1210,7.8700,1721
2,122639,Parag Parikh Flexi Cap Fund - Direct Plan - Growth,2019-05-23,7.0000,90.6579,2026-05-18,95.7732,85.2599,6.3300,1719


---
## 12. Cross-file Consistency Checks
After cleaning each file individually, verify that `scheme_code` values are consistent across all files — since `scheme_code` is the **primary join key** for your entire dashboard.

In [284]:
# ── Cross-file scheme_code coverage ──
codes = {
    'all_schemes':      set(df_schemes['scheme_code']),
    'amfi_master':      set(df_amfi['scheme_code']),
    'nav_history_raw':  set(df_nav['scheme_code']),
    'nav_snapshots':    set(df_snap['scheme_code']),
    'scheme_metadata':  set(df_meta['scheme_code']),
    'scheme_profiles':  set(df_prof['scheme_code']),
}

print('Scheme code counts per file:')
for name, s in codes.items():
    print(f'  {name}: {len(s)} unique codes')

Scheme code counts per file:
  all_schemes: 37602 unique codes
  amfi_master: 14367 unique codes
  nav_history_raw: 14 unique codes
  nav_snapshots: 14 unique codes
  scheme_metadata: 14 unique codes
  scheme_profiles: 14 unique codes


In [285]:
# ── Check which codes in nav_history_raw are NOT in all_schemes (orphan codes) ──
orphan_nav = codes['nav_history_raw'] - codes['all_schemes']
print(f'NAV codes not in all_schemes (orphans): {len(orphan_nav)}')
if orphan_nav:
    print(sorted(orphan_nav))

NAV codes not in all_schemes (orphans): 0


In [286]:
# ── Check which codes in scheme_metadata are in nav_history ──
meta_in_nav = codes['scheme_metadata'] & codes['nav_history_raw']
print(f'scheme_metadata codes present in nav_history: {len(meta_in_nav)} / {len(codes["scheme_metadata"])}')

scheme_metadata codes present in nav_history: 14 / 14


---
## 13. Save All Cleaned Files
Save each cleaned DataFrame as a new CSV so you never touch the raw data again.

In [287]:
import os
os.makedirs('data/cleaned', exist_ok=True)

df_schemes.to_csv('data/cleaned/all_schemes_clean.csv',         index=False)
df_amfi.to_csv('data/cleaned/amfi_master_clean.csv',            index=False)
df_aum.to_csv('data/cleaned/aum_amfi_monthly_clean.csv',        index=False)
df_bench.to_csv('data/cleaned/benchmark_data_clean.csv',        index=False)
df_cat.to_csv('data/cleaned/category_reference_clean.csv',      index=False)
df_nav.to_csv('data/cleaned/nav_history_clean.csv',             index=False)
df_snap.to_csv('data/cleaned/nav_snapshots_clean.csv',          index=False)
df_meta.to_csv('data/cleaned/scheme_metadata_clean.csv',        index=False)
df_prof.to_csv('data/cleaned/scheme_profiles_clean.csv',        index=False)

print('All cleaned files saved to /cleaned/ ✅')

All cleaned files saved to /cleaned/ ✅


---
## Summary of All Cleaning Operations

| File | Rows | Key Issues Fixed | Nulls Handled | Date Cols Fixed |
|------|------|-----------------|---------------|-----------------|
| `all_schemes.csv` | 37,601 | Trimmed 274 whitespace names | ISIN nulls are valid — kept | — |
| `amfi_master.csv` | 14,366 | Replaced `'-'` with NaN in ISIN cols | — | `nav_date` → datetime |
| `aum_amfi_monthly.csv` | 91→~45 data rows | Re-read with correct header, renamed 11 cols, removed group/blank rows, converted all numerics | Group rows removed | — |
| `aum_and_ter.csv` | 14 | All 3 AUM/TER cols are null (fallback source) — flagged, not dropped. Added `nav_available` flag | Flagged | `latest_nav_date` → datetime |
| `benchmark_data.csv` | 1,812 | Sorted by date | Holiday gaps (NaN) kept | `date` → datetime |
| `category_reference.csv` | 41 | Trimmed strings | None | — |
| `nav_history_raw.csv` | 22,692 | Sorted by (scheme_code, date). Validated NAV > 0 | None | `date` → datetime |
| `nav_snapshots.csv` | 13 | Trimmed fund_name | None | — |
| `scheme_metadata.csv` | 14 | Flagged 1 incomplete row (scheme 119260) with `metadata_complete` column | 1 row has 6 null cols — flagged not dropped | — |
| `scheme_profiles.csv` | 13 | Logical validation of 52w high/low. Trimmed fund_name | None | `inception_date`, `latest_nav_date` → datetime |
| `amapr2026repo.xls` | Needs `xlrd` | Trim cols & strings, drop full-row dups per sheet | Varies by sheet | Varies by sheet |
